In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders.image import UnstructuredImageLoader
from langchain_unstructured import UnstructuredLoader


In [ ]:
load_dotenv()

#create llm object
llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash")

loader = UnstructuredLoader("sample_unstructured_test.pdf")
documents = loader.load()

In [ ]:
#split the text using recursive approch
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)
chunks = text_splitter.split_documents(documents)

In [ ]:
#Embedding model 
embedding=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

#------------------generate embeddings----------------------
embeddings=embedding.embed_documents([text.page_content for text in chunks])   

#create database from chunks and embedding model                                                           
db=FAISS.from_documents(chunks,embedding)                                       

In [ ]:
db_save_path = "db/faiss_index_gemini"
db.save_local(db_save_path)
loaded_db=FAISS.load_local(db_save_path,embedding,allow_dangerous_deserialization=True)
question="What is the backend for the project?"
similler_data=loaded_db.similarity_search(question,k=2)

In [ ]:

# prompt template and generation                                                                
prompt_template = """
Use the following context to answer the question:

{context}

Question: {question}

Answer:
"""

prompt=PromptTemplate.from_template(prompt_template)
formatted_prompt=prompt.invoke({
    "context":similler_data[0].page_content,
    "question":question
})
# print(formatted_prompt)
final_response=llm.invoke(formatted_prompt)
print(final_response.content)

In [ ]:
# from langchain_unstructured import UnstructuredLoader

# loader = UnstructuredLoader("sample_unstructured_test.pdf")
# documents = loader.load()
# for element in documents:
#     print(element.page_content)
#     print(element.metadata.get("category"))

from langchain_community.document_loaders.image import UnstructuredImageLoader

loader = UnstructuredImageLoader("sc.jpg")
image_docs = loader.load()

In [ ]:
# Unstructured for partitioning different file types
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx
from unstructured.partition.image import partition_image
from unstructured.partition.text import partition_text
from unstructured.documents.elements import Image
import os
PROCESSOR_MAP = {
    ".pdf": partition_pdf,
    ".docx": partition_docx,
    ".txt": partition_text,
    ".jpg": partition_image,
    ".jpeg": partition_image,
    ".png": partition_image,
}

In [ ]:
file_path = "data/my_document.pdf"

print(f"Partitioning document: {file_path}")    
_, file_extension = os.path.splitext(file_path)
file_extension = file_extension.lower()

processor = PROCESSOR_MAP.get(file_extension)

if not processor:
    raise ValueError(f"No processor found for file type: {file_extension}")

# Special handling for images and PDFs to extract image data
if processor == partition_image:
    result=processor(file_path)
elif processor == partition_pdf:
    # This tells the PDF partitioner to also extract embedded images
    result=processor(file_path, extract_images_in_pdf=True, infer_table_structure=True)
    for i, element in enumerate(result, 1):
        # print('unstructured.documents.elements.Text' in str(type(element)))
        if isinstance(element,Image):
            print("That is a fucking image")
            image_path=element.metadata.image_path
            if image_path and os.path.exists(image_path):
                with open(image_path, "rb") as img_file:
                    image_bytes = img_file.read()
                    print(f"Image {i} size (bytes): {len(image_bytes)},{image_bytes}")
        else:
            print(element.text)

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
db = FAISS.load_local("db/faiss_index_multimodal", embedding_model, allow_dangerous_deserialization=True)

# docs = db._documents
# print(f"Total documents stored: {len(docs)}")

# for i, doc in enumerate(docs[:5]):  # show first 5
#     print(f"\n--- Document #{i+1} ---")
#     print(f"Page content:\n{doc.page_content}")
#     print(f"Metadata:\n{doc.metadata}")


RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss-wheels\faiss-wheels\faiss\faiss\impl\io.cpp:68: Error: 'f' failed: could not open db\faiss_index_multimodal\index.faiss for reading: No such file or directory